In [2]:
import os
import numpy as np
import torch
from datasets import ClassLabel, Dataset
from torch import nn
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)
# Try one of these in place of MODEL_NAME:
#MODEL_NAME = "microsoft/deberta-v3-small"
MODEL_NAME = "BAAI/bge-base-en-v1.5"
#MODEL_NAME = "roberta-base"
OUTPUT_DIR = "./bge-base-en-v1.5-qa-scorer"



SEED = 42

PROMPT_TYPES = [
    "Zero shot",
    "One shot",
    "Few shot",
    "Adaptive",
    "Prompt chaining",
    "Self consistency prompting",
    "Contrastive prompting",
    "Meta prompting",
    "Reverse prompting",
]


def build_raw_samples() -> list[dict]:
    """
    54 records (6 per prompting technique across high, medium, and low score tiers)
    evaluating conceptual "What is ... prompting?" answers on a 0.0 - 10.0 scale.
    """
    return [
        # 1. Zero shot (6 samples)
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting asks a model to perform a task directly from instructions alone without providing any labeled examples or demonstrations in the prompt.",
            "score": 9.5,
        },
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "In zero shot prompting, the model relies on its pre-trained knowledge and task instructions with zero in-context examples included.",
            "score": 9.5,
        },
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting gives the model a few examples and asks for a rough guess.",
            "score": 1.0,
        },
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting is when you don't write a prompt and let the model read your mind.",
            "score": 0,
        },
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "It means giving the model ten solved examples before asking a question.",
            "score": 1.0,
        },
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting sets temperature to zero so the output never changes.",
            "score": 0.0,
        },
        # 2. One shot (6 samples)
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting provides a single example input-output demonstration alongside the task instructions so the model sees the expected format or pattern before answering.",
            "score": 9.8,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting includes one sample pair in the prompt to illustrate the desired output structure before the model answers the real query.",
            "score": 9.8,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting uses examples in a prompt to help the model understand the task.",
            "score": 7.2,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting adds a single example, though it doesn't mention how it guides output format.",
            "score": 8.6,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting runs the prompt once every ten minutes on a cron job.",
            "score": 0.0,
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting means you can only send one message per day to the API.",
            "score": 0.0,
        },
        # 3. Few shot (6 samples)
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting supplies multiple diverse examples in the context window to guide formatting, edge cases, and reasoning style for a target query without weight updates.",
            "score": 9.1,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting provides several input-output examples inside the prompt so the model generalizes the pattern in-context without fine-tuning.",
            "score": 8.9,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting adds examples to the prompt, though it is similar to one shot prompting.",
            "score": 8.5,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting gives multiple examples to show the pattern, without explaining edge cases or no-fine-tuning.",
            "score": 7.0,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting retrains all model weights on a small dataset.",
            "score": 0.0,
        },
        {
            "prompt_type": "Few shot",
            "question": "What is few shot prompting?",
            "response": "Few shot prompting means firing a model after a few bad responses.",
            "score": 0.0,
        },
        # 4. Adaptive (6 samples)
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting dynamically adjusts prompt phrasing, detail level, or strategy based on user expertise, task difficulty, or intermediate model feedback.",
            "score": 9.0,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting tailors instructions, complexity, or reasoning steps on the fly in response to context, difficulty, or prior outputs.",
            "score": 8.7,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting changes how a prompt is written depending on the situation.",
            "score": 0.5,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting adapts the tone of a reply, without covering difficulty or feedback loops.",
            "score": 0.2,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting uses the exact same rigid template for every user and every question.",
            "score": 0.0,
        },
        {
            "prompt_type": "Adaptive",
            "question": "What is adaptive prompting?",
            "response": "Adaptive prompting adjusts the monitor stand height for ergonomics.",
            "score": 0.0,
        },
        # 5. Prompt chaining (6 samples)
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining breaks a complex task into sequential sub-prompts where the output of one step becomes the structured input to the next step.",
            "score": 9.4,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining decomposes multi-stage workflows into smaller prompts passed sequentially so each step stays focused and verifiable.",
            "score": 8.9,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining splits a big task into multiple prompts instead of using one giant prompt.",
            "score": 8.5,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining uses multiple prompts in order, though it doesn't mention passing outputs forward.",
            "score": 3.1,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining links several prompts together with physical metal chains.",
            "score": 0.5,
        },
        {
            "prompt_type": "Prompt chaining",
            "question": "What is prompt chaining?",
            "response": "Prompt chaining locks the model into repeating the same sentence forever.",
            "score": 0.0,
        },
        # 6. Self consistency prompting (6 samples)
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting samples multiple diverse reasoning paths for the same prompt and selects the most consistent final answer—often via majority vote—rather than relying on a single greedy path.",
            "score": 9.5,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting generates several reasoning traces and aggregates their final answers by majority consensus to improve reliability on multi-step tasks.",
            "score": 9.0,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting tries more than one reasoning attempt to see which answer comes up most.",
            "score": 9.8,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting runs a prompt multiple times to check for agreement.",
            "score": 2.0,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting asks the model to agree with whatever false claim the user makes.",
            "score": 0.0,
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "Self consistency prompting forces the model to give the exact same typo every time.",
            "score": 0.0,
        },
        # 7. Contrastive prompting (6 samples)
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting shows both desirable and undesirable examples—or contrasts valid vs. invalid reasoning—so the model learns clear boundaries for what to emulate and what to avoid.",
            "score": 9.2,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting pairs positive and negative demonstrations—or contrasting reasoning paths—to clarify what behavior to follow and what pitfalls to reject.",
            "score": 9.8,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting compares good and bad examples in the prompt.",
            "score": 3.5,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting includes negative examples to warn the model what not to do.",
            "score": 8.3,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting only changes screen brightness and contrast settings.",
            "score": 0.0,
        },
        {
            "prompt_type": "Contrastive prompting",
            "question": "What is contrastive prompting?",
            "response": "Contrastive prompting means arguing against the user in every reply.",
            "score": 0.0,
        },
        # 8. Meta prompting (6 samples)
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting uses a model to design, critique, refine, or generate prompts, rubrics, or problem-solving templates for itself or another workflow.",
            "score": 9.1,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting treats prompting itself as the task—having an LLM write, optimize, or structure prompts and evaluation criteria for downstream queries.",
            "score": 8.9,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting involves using AI to help write or improve prompts.",
            "score": 2.4,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting asks the model to review a prompt's wording without mentioning templates or workflows.",
            "score": 0.0,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting is only used when talking about Facebook company news.",
            "score": 0.0,
        },
        {
            "prompt_type": "Meta prompting",
            "question": "What is meta prompting?",
            "response": "Meta prompting puts metadata tags inside HTML headers for search engines.",
            "score": 0.2,
        },
        # 9. Reverse prompting (6 samples)
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting flips the interaction so the model asks targeted clarifying questions to uncover missing requirements before generating a final solution.",
            "score": 9.3,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting has the assistant interview the user with focused questions to gather constraints and intent before answering.",
            "score": 8.8,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting has the model ask questions back to the user before answering.",
            "score": 0.0,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting means asking the model to double-check its answer after responding.",
            "score": 0.0,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting spells every word in the prompt backward from right to left.",
            "score": 0.0,
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting sends the response first and the question afterward.",
            "score": 0.0,
        },
    ]


def score_to_centered(score_0_10: float) -> float:
    """Map [0.0, 10.0] -> [-1.0, 1.0] so 5.0 sits at 0.0."""
    clamped = float(np.clip(score_0_10, 0.0, 10.0))
    return (clamped - 5.0) / 5.0


def centered_to_score(pred_centered: float) -> float:
    """Map [-1.0, 1.0] -> [0.0, 10.0]."""
    score = (float(pred_centered) * 5.0) + 5.0
    return float(np.clip(score, 0.0, 10.0))


def build_formatted_dataset() -> Dataset:
    rows = []
    for idx, item in enumerate(build_raw_samples()):
        text = (
            f"Prompting Topic: {item['prompt_type']}\n"
            f"Question: {item['question']}\n"
            f"Response: {item['response']}"
        )
        rows.append(
            {
                "id": idx,
                "prompt_type": item["prompt_type"],
                "text": text,
                "label": score_to_centered(item["score"]),
            }
        )
    return Dataset.from_list(rows)


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds_10 = np.array([centered_to_score(p) for p in preds.squeeze(-1)])
    labels_10 = np.array([centered_to_score(y) for y in labels.squeeze()])
    mse = float(np.mean((labels_10 - preds_10) ** 2))
    mae = float(np.mean(np.abs(labels_10 - preds_10)))
    pred_std = float(np.std(preds_10))
    return {
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
        "mae": mae,
        "pred_std": pred_std,
    }


def get_precision_flags() -> dict:
    """Configure precision cleanly across CPU and CUDA GPUs."""
    if not torch.cuda.is_available():
        return {"use_cpu": True, "bf16": False, "fp16": False}
    bf16_supported = torch.cuda.is_bf16_supported()
    return {
        "use_cpu": False,
        "bf16": bf16_supported,
        "fp16": False,
    }


class CenteredRegressionTrainer(Trainer):
    """Uses SmoothL1Loss to stabilize regression on small datasets."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float().view(-1)
        outputs = model(**inputs)
        logits = outputs.logits.float().view(-1)
        loss = nn.SmoothL1Loss(beta=0.2)(logits, labels)
        return (loss, outputs) if return_outputs else loss


def train_model():
    set_seed(SEED)
    precision = get_precision_flags()
    device_label = "GPU (CUDA)" if not precision["use_cpu"] else "CPU"
    print(f"Training on: {device_label} | precision={precision}")

    dataset = build_formatted_dataset().cast_column(
        "prompt_type", ClassLabel(names=PROMPT_TYPES)
    )
    split = dataset.train_test_split(
        test_size=0.2,
        seed=SEED,
        stratify_by_column="prompt_type",
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=512,
            padding="max_length",
        )

    tokenized_train = split["train"].map(tokenize_fn, batched=True)
    tokenized_eval = split["test"].map(tokenize_fn, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
        problem_type="regression",
    )

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=8,
        weight_decay=0.01,
        warmup_steps=0.05,
        max_grad_norm=1.0,
        load_best_model_at_end=True,
        metric_for_best_model="mae",
        greater_is_better=False,
        logging_steps=5,
        report_to="none",
        seed=SEED,
        use_cpu=precision["use_cpu"],
        bf16=precision["bf16"],
        fp16=precision["fp16"],
    )

    trainer = CenteredRegressionTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    eval_results = trainer.evaluate()
    print("\nValidation metrics (0-10 scale):", eval_results)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Saved model and tokenizer to {OUTPUT_DIR}")


def score_responses(model_dir: str, test_cases: list[dict]):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()

    print("\n--- Post-Training Sample Evaluations (Scale: 0 - 10) ---")
    for i, case in enumerate(test_cases, start=1):
        text = (
            f"Prompting Topic: {case['prompt_type']}\n"
            f"Question: {case['question']}\n"
            f"Response: {case['response']}"
        )
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            pred_centered = model(**inputs).logits.squeeze().item()

        final_score = centered_to_score(pred_centered)
        print(f"\nSample {i} [{case['prompt_type']}]")
        print(f"  Question : {case['question']}")
        print(f"  Response : {case['response']}")
        print(f"  Score    : {final_score:.2f} / 10.00")


if __name__ == "__main__":
    train_model()

    sample_tests = [
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting asks a model to complete a task directly from instructions without any examples included in the prompt.",
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting gives the model one example of how to map an input to an output before asking it to handle a new input.",
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "It deletes the system prompt and randomly drops words from the user question.",
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting has the assistant ask clarifying questions to gather missing context before delivering the final answer.",
        },
    ]

    score_responses(OUTPUT_DIR, sample_tests)





Training on: CPU | precision={'use_cpu': True, 'bf16': False, 'fp16': False}


Casting the dataset:   0%|          | 0/54 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: BAAI/bge-base-en-v1.5
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (11 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model and tokenizer to ./bge-base-en-v1.5-qa-scorer


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Post-Training Sample Evaluations (Scale: 0 - 10) ---

Sample 1 [Zero shot]
  Question : What is zero shot prompting?
  Response : Zero shot prompting asks a model to complete a task directly from instructions without any examples included in the prompt.
  Score    : 0.72 / 10.00

Sample 2 [One shot]
  Question : What is one shot prompting?
  Response : One shot prompting gives the model one example of how to map an input to an output before asking it to handle a new input.
  Score    : 5.34 / 10.00

Sample 3 [Self consistency prompting]
  Question : What is self consistency prompting?
  Response : It deletes the system prompt and randomly drops words from the user question.
  Score    : 0.00 / 10.00

Sample 4 [Reverse prompting]
  Question : What is reverse prompting?
  Response : Reverse prompting has the assistant ask clarifying questions to gather missing context before delivering the final answer.
  Score    : 1.17 / 10.00


In [ ]:
    train_model()



In [ ]:
sample_tests = [
        {
            "prompt_type": "Zero shot",
            "question": "What is zero shot prompting?",
            "response": "Zero shot prompting asks a model to complete a task directly from instructions without any examples included in the prompt.",
        },
        {
            "prompt_type": "One shot",
            "question": "What is one shot prompting?",
            "response": "One shot prompting gives the model one example of how to map an input to an output before asking it to handle a new input.",
        },
        {
            "prompt_type": "Self consistency prompting",
            "question": "What is self consistency prompting?",
            "response": "It deletes the system prompt and randomly drops words from the user question.",
        },
        {
            "prompt_type": "Reverse prompting",
            "question": "What is reverse prompting?",
            "response": "Reverse prompting has the assistant ask clarifying questions to gather missing context before delivering the final answer.",
        },
    ]



In [3]:
score_responses(OUTPUT_DIR, sample_tests)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Post-Training Sample Evaluations (Scale: 0 - 10) ---

Sample 1 [Zero shot]
  Question : What is zero shot prompting?
  Response : Zero shot prompting asks a model to complete a task directly from instructions without any examples included in the prompt.
  Score    : 6.26 / 10.00

Sample 2 [One shot]
  Question : What is one shot prompting?
  Response : One shot prompting gives the model one example of how to map an input to an output before asking it to handle a new input.
  Score    : 7.19 / 10.00

Sample 3 [Self consistency prompting]
  Question : What is self consistency prompting?
  Response : It deletes the system prompt and randomly drops words from the user question.
  Score    : 2.86 / 10.00

Sample 4 [Reverse prompting]
  Question : What is reverse prompting?
  Response : Reverse prompting has the assistant ask clarifying questions to gather missing context before delivering the final answer.
  Score    : 7.89 / 10.00
